# Importing Necessary Packages 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, StringType, DoubleType

# Fetching All Data from transformed tables

In [0]:
exchange_rates_df = spark.table("02_dev_silver.transformed.exchange_rates")
customers_df=spark.table("02_dev_silver.transformed.customers")
order_items_df = spark.table("02_dev_silver.transformed.order_items")
orders_df = spark.table("02_dev_silver.transformed.orders")
products_df = spark.table("02_dev_silver.transformed.products")

# Creating dimension table for date

In [0]:
#Combining all the dates from orders and customers table
order_dates = orders_df.select(col("order_date").alias("date"))
customer_dates = customers_df.select(col("registration_date").alias("date"))
dim_date = order_dates.union(customer_dates).dropna().dropDuplicates()

#Creating a surrogate key for date
dim_date = dim_date.withColumn(
    "date_key",
    date_format(col("date"), "yyyyMMdd").cast("int")
)

#Defining the order of the columns
dim_date = dim_date.select(
    "date_key",
    col("date").alias("full_date"),
)

#Adding a row for unknown date
unknown_date_schema = StructType([
    StructField("date_key", IntegerType(), False),
    StructField("full_date", DateType(), True)
])
unknown_date = spark.createDataFrame([
    Row(date_key=-1, full_date=None)
], schema=unknown_date_schema)
dim_date = dim_date.union(unknown_date)

# Creating dimension table for customer

In [0]:
#Creating a surrogate key for customers
window=Window.orderBy("customer_id")
dim_customer = customers_df.withColumn(
    "customer_key", row_number().over(window)
).select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email_address",
    "customer_country",
    "channel",
    "registration_date"
)

#Combining the registration date with the dim_date table
dim_customer = dim_customer.join(
    dim_date.select(col("full_date").alias("registration_date"), "date_key"),
    "registration_date",
    "left"
).withColumn(
    "registration_date_key",
    when(col("date_key").isNull(), -1).otherwise(col("date_key"))
)

#Defining the order of the columns
dim_customer = dim_customer.select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email_address",
    "customer_country",
    "channel",
    col("registration_date_key")
)

#Adding a row for unknown date
unknown_customer_schema = StructType([
    StructField("customer_key", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email_address", StringType(), True),
    StructField("customer_country", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("registration_date_key", IntegerType(),True),
])
unknown_customer = spark.createDataFrame([
    Row(
        customer_key=-1,
        customer_id="Unknown",
        customer_name="Unknown",
        email_address="Unknown",
        customer_country="Unknown",
        channel="Unknown",
        registration_date_key=-1
    )
], schema=unknown_customer_schema)
dim_customer = dim_customer.union(unknown_customer)

# Creating dimension table for product

In [0]:
#Creating a surrogate key for customers
dim_product = products_df.withColumn(
    "product_key", row_number().over(Window.orderBy("product_id"))
)

#Converting Country code to country name
dim_product=dim_product.withColumn("order_country",
when(col("country_code")=="UK", "United Kingdom")
.when(col("country_code")=="DE", "Germany")
.when(col("country_code")=="ES", "Spain")
.when(col("country_code")=="IN", "India")
.when(col("country_code")=="CN", "China")
.otherwise(col("country_code")))

#Defining the order of the columns
dim_product=dim_product.select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "price",
    "currency",
    "order_country",
    "exchange_rate_to_usd",
    "base_price"
)

#Adding a row for unknown product
unknown_product_schema = StructType([
    StructField("product_key", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("order_country", StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True),
    StructField("base_price", DoubleType(), True)
])
unknown_product = spark.createDataFrame([
    Row(
        product_key=-1,
        product_id="Unknown",
        product_name="Unknown",
        category="Unknown",
        price=None,
        currency="Unknown",
        order_country="Unknown",
        exchange_rate_to_usd=None,
        base_price=None
    )
], schema=unknown_product_schema)
dim_product = dim_product.union(unknown_product)

# Creating fact table based on order items

In [0]:
#Joining all the necessary tables to create fact table
fact_df = order_items_df.join(
    orders_df.select("order_id", "customer_id", "order_date", "order_status","channel"),
    "order_id",
    "left"
)
fact_df = fact_df.join(
    dim_customer.select("customer_id", "customer_key"),
    "customer_id",
    "left"
)
fact_df = fact_df.join(
    dim_product.select("product_id", "product_key"),
    "product_id",
    "left"
)

#Handling null in surrogate keys
fact_df = fact_df.withColumn(
    "date_key",
    when(col("order_date").isNull(), -1)
    .otherwise(date_format(col("order_date"), "yyyyMMdd").cast("int"))
).withColumn(
    "customer_key",
    when(col("customer_key").isNull(), -1).otherwise(col("customer_key"))
).withColumn(
    "product_key",
    when(col("product_key").isNull(), -1).otherwise(col("product_key"))
)

#Defining the order of the columns
fact_order_items = fact_df.select(
    "order_item_id",
    "order_id",
    "customer_key",
    "product_key",
    "date_key",
    "quantity",
    "base_unit_price",
    "base_line_total",
    "order_status",
    col("channel").alias("order_channel")
)

# Saving all the tables

In [0]:
table_names = {
    "02_dev_silver.facts_and_dims.dim_customer": dim_customer,
    "02_dev_silver.facts_and_dims.dim_product": dim_product,
    "02_dev_silver.facts_and_dims.dim_date": dim_date,
    "02_dev_silver.facts_and_dims.fact_order_items": fact_order_items
}

for table_name, df in table_names.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)